# MediaPipe + Random Forest Emotion Classifier Training

This notebook trains a Random Forest model on MediaPipe Face Mesh landmarks, matching the state-of-the-art tele-rehabilitation approach outlined in the case study.


In [ ]:
import os
import cv2
import numpy as np
import pickle
from tqdm import tqdm
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
try:
    from mediapipe import solutions
    mp_face_mesh = solutions.face_mesh
except (ImportError, AttributeError):
    import mediapipe as mp
    mp_face_mesh = mp.solutions.face_mesh
import matplotlib.pyplot as plt
import sys
from imblearn.under_sampling import RandomUnderSampler

# Local imports
sys.path.append(os.path.abspath('..'))
from realtime.multi_emotion_predictor import LandmarkFeatureExtractor
from model.config import EMOTION_LABELS, DATA_DIR, MODEL_SAVE_PATH

face_mesh = mp_face_mesh.FaceMesh(static_image_mode=True, max_num_faces=1, min_detection_confidence=0.5)
feature_extractor = LandmarkFeatureExtractor('empiric')

def extract_features_from_image(img_path):
    img = cv2.imread(img_path)
    if img is None: return None
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(img_rgb)
    
    if not results.multi_face_landmarks:
        return None
        
    landmarks = results.multi_face_landmarks[0]
    landmarks_3d = np.array([[lm.x, lm.y, lm.z] for lm in landmarks.landmark])
    
    # Normalize landmarks
    ref_point = landmarks_3d[0]
    norm_landmarks = landmarks_3d - ref_point
    max_vals = np.max(np.abs(norm_landmarks), axis=0)
    max_vals[max_vals == 0] = 1e-6
    norm_landmarks = norm_landmarks / max_vals
    
    # Extract
    features = feature_extractor.extract_features(norm_landmarks)
    return features

def prepare_dataset(base_dir):
    X, y = [], []
    classes = sorted(EMOTION_LABELS)
    for class_idx, class_name in enumerate(classes):
        class_dir = os.path.join(base_dir, class_name)
        if not os.path.isdir(class_dir):
            continue
        print(f"Processing {class_name}...")
        for img_name in tqdm(os.listdir(class_dir)[:500]): # Limit for fast training in "we have no time" scenario, remove limit for prod!
            img_path = os.path.join(class_dir, img_name)
            features = extract_features_from_image(img_path)
            if features is not None:
                X.append(features)
                y.append(class_idx)
    return np.array(X), np.array(y)

from imblearn.over_sampling import SMOTE

# Extract Features
print("Preparing Training Set...")
X_train, y_train = prepare_dataset(os.path.join(DATA_DIR, 'train'))
print("Preparing Test Set...")
X_test, y_test = prepare_dataset(os.path.join(DATA_DIR, 'test'))

# Scale up Disgust, Fear, Sad using SMOTE Over-sampling for production-grade balancing
smote = SMOTE(sampling_strategy='auto', random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f"Train shapes: {X_train_res.shape}, {y_train_res.shape}")
print(f"Test shapes: {X_test.shape}, {y_test.shape}")


In [ ]:
\
# Train Random Forest prioritizing Disgust, Fear, Sad
print("Training Random Forest with intense class weights...")
# Emphasize our target classes mathematically even after SMOTE
class_weights = {
    0: 1.0,  # Angry
    1: 4.0,  # Disgust (Extreme focus)
    2: 2.5,  # Fear (Focus)
    3: 1.0,  # Happy
    4: 1.0,  # Neutral
    5: 2.5,  # Sad (Focus)
    6: 1.0   # Surprise
}

rf_model = RandomForestClassifier(
    n_estimators=200, 
    max_depth=25, 
    class_weight=class_weights, 
    random_state=42, 
    n_jobs=-1
)
rf_model.fit(X_train_res, y_train_res)

# Evaluate
print("Evaluating Model...")
y_pred = rf_model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=EMOTION_LABELS))

# Save the model
save_path = os.path.join('..', 'trained_models', 'emotion_rf_model.pkl')
os.makedirs(os.path.dirname(save_path), exist_ok=True)
with open(save_path, 'wb') as f:
    pickle.dump(rf_model, f)
print(f"Model saved to {save_path}")
